[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Jev_CeNN_All28_Substitution_LR_0p01_to_0p002_V6_Colab.ipynb)

# Jev-CeNN V6 — All-28 Attention Substitution: joint training LR 0.01 → 0.002, FFNs preserved

**This is a new full-model experiment and Hugging Face export, not a change to V4/V5.**

- **All 28 attention modules are replaced, always.** Original pretrained FFN weights, FFN/attention LayerNorms, original residual connections and task head remain bit-identical. No acceptance gates or block skipping.
- **Training:** joint end-to-end substitution learning on all 28 CeNN branches. Exactly requested **learning rate 0.01 → 0.002** (cosine-annealed), for *CeNN correction parameters*. Pretrained QKV/output projections are optional and use a separate tiny learning rate (1e-5 → 2e-6) to prevent destructive updates. Frozen FFNs and head have no gradients.
- **Efficient training:** teacher logits cached once, teacher moved to CPU, CUDA gradient accumulation + mixed precision, original task-head dropout disabled, periodic attention-weight snapshots. Joint teacher-logit KL + gold-label cross-entropy.
- **Initialization:** use previously trained all-28 V4 checkpoints from `all28_results.zip` (or V5 Hugging Face repo). If neither is available, initialize all 28 new attention modules directly from original model weights with a conservative default window and train them. This fallback is NOT expected to reproduce your V4 94.8% agreement before training.
- **Model output:** a complete new model with full safetensors weights, tokenizer, custom attention loader, training report and private-by-default Hugging Face upload. Standalone reload parity checked before upload. Existing model remains untouched.

## What to upload
Place your prior V4 `all28_results.zip` in Colab Files at `/content/all28_results.zip`, or set `INITIAL_HF_REPO` to your already-uploaded V5 model ID. When checkpoints already exist, there is **no independent per-layer retraining**: V6 jointly trains the model you already converted.

**Important:** The requested 0.01 learning rate is high for pretrained projection matrices; the notebook does NOT apply it directly to those matrices. New CeNN paths can make inference slower; full-model speed and gold accuracy are measured, not assumed.


In [1]:
#@title 1. Setup: 28 layers, cosine 0.01 → 0.002 substitution LR
%pip -q install "transformers==4.57.6" "datasets>=2.20,<5" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install -U "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@eba220d8131dbdf06f748545a21831327e5ad37d"

from pathlib import Path
import copy, gc, hashlib, json, math, os, random, shutil, sys, time, zipfile
from contextlib import nullcontext
from datetime import datetime,timezone
import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import HfApi, snapshot_download, get_token, login
from safetensors.torch import save_model
from transformers import AutoTokenizer
from tinycenn_lm.standalone_decision import (
    QTYPES, build_checkpoint_model, build_sequence,
    collate_items, normalize_question, full_attention_indices
)
from tinycenn_lm.decision_training import require_finite
from tinycenn_lm.all28_hf import (
    WindowCeNNAttention, build_all28_from_dir, load_all28
)
import tinycenn_lm.all28_hf as model_code

SEED=2028
SOURCE_MODEL="convaiinnovations/laya-typed-decisions"
INITIAL_HF_REPO="" #@param {type:"string"}
V4_ZIP_PATH="/content/all28_results.zip" #@param {type:"string"}
V4_DIR="/content/Jev_CeNN_All28_AttentionOnly_BruteForce_V4" #@param {type:"string"}
# Joint optimization: 0.01 -> 0.002 for newly enabled CeNN branches.
LR_START=0.01 #@param {type:"number"}
LR_END=0.002 #@param {type:"number"}
PROJECTION_LR_SCALE=0.001 #@param {type:"number"}
JOINT_STEPS=300 #@param {type:"integer"}
BATCH=2 #@param {type:"integer"}
GRAD_ACCUM=4 #@param {type:"integer"}
WARMUP_STEPS=0 #@param {type:"integer"}
EVAL_EVERY=50 #@param {type:"integer"}
SAVE_EVERY=100 #@param {type:"integer"}
TEACHER_TEMPERATURE=2.0
GOLD_WEIGHT=0.20
KL_WEIGHT=1.0
MAX_GRAD_NORM=0.5
MAX_LEN=512
TRAIN_CASES=700
DEV_CASES=220
HOLDOUT_CASES=200
PROBE_CASES=80
PROBE_BATCHES=12  # Required by shared dataset-split sanity check
DEV_GATE_DECISIONS=240
HOLDOUT_DECISIONS=500
TRAIN_DECISIONS_LIMIT=1200 #@param {type:"integer"}
ENABLE_CENN_ALL=True #@param {type:"boolean"}
TRAIN_PRETRAINED_QKV=True #@param {type:"boolean"}
AMP=True #@param {type:"boolean"}
BENCH_REPEATS=15
UPLOAD_TO_HF=True #@param {type:"boolean"}
HF_PRIVATE=True #@param {type:"boolean"}
HF_REPO_NAME="jev-cenn-all28-substitution-lr001-lr002-v6" #@param {type:"string"}
OUTPUT_DIR=Path("/content/Jev_CeNN_All28_Substitution_V6")
EXPORT_DIR=OUTPUT_DIR/"huggingface_model"
STAGE_DIR=OUTPUT_DIR/"checkpoints"
EXPORT_DIR.mkdir(parents=True,exist_ok=True)
STAGE_DIR.mkdir(parents=True,exist_ok=True)
assert 0<LR_END<LR_START and LR_START==0.01 and LR_END==0.002
assert JOINT_STEPS>0 and GRAD_ACCUM>=1 and BATCH>=1 and EVAL_EVERY>=1
assert 0<=WARMUP_STEPS<JOINT_STEPS
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab CUDA GPU runtime for joint 28-layer training.")
torch.cuda.manual_seed_all(SEED)
device=torch.device("cuda")
amp_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16
use_scaler=bool(AMP and amp_dtype==torch.float16)
# No gradient checkpointing by default; enable below only if needed for GPU RAM.
ENABLE_GRADIENT_CHECKPOINTING=True
print("GPU",torch.cuda.get_device_name(0),"AMP",amp_dtype,
      "CeNN schedule",LR_START,"->",LR_END,"QKV scale",PROJECTION_LR_SCALE)
source_dir=Path(snapshot_download(
    SOURCE_MODEL,allow_patterns=[
      "rl_agent_config.json","model.safetensors","encoder/*","tokenizer/*"]))
teacher,source_cfg=build_checkpoint_model(source_dir)
student_eval,_=build_checkpoint_model(source_dir)
teacher.eval().requires_grad_(False)
student_eval.eval().requires_grad_(False)
teacher.encoder.config.reference_compile=False
student_eval.encoder.config.reference_compile=False
tokenizer=AutoTokenizer.from_pretrained(str(source_dir/"tokenizer"))
GLOBAL_LAYERS=tuple(full_attention_indices(teacher))
assert GLOBAL_LAYERS==tuple(range(0,28,3))
assert len(teacher.encoder.layers)==len(student_eval.encoder.layers)==28
print("Will train ALL 28 attention substitutions; original FFNs remain frozen.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 68.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
GPU Tesla T4 AMP torch.float16 CeNN schedule 0.01 -> 0.002 QKV scale 0.001


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

rl_agent_config.json:   0%|          | 0.00/847 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/337 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/843M [00:00<?, ?B/s]

Will train ALL 28 attention substitutions; original FFNs remain frozen.


In [2]:
#@title 3. Partition dataset by CASE: TRAIN, PROBE, DEV-GATE and sealed HOLDOUT
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

from tinycenn_lm.jev_cenn_selective import (
    verify_case_partition, judge_replacement, plan_active_layers,
)
rows=list(load_dataset("LocalLLaMA/typed-decisions","all",split="train"))
rng=random.Random(SEED)
rng.shuffle(rows)
assert TRAIN_CASES+DEV_CASES+HOLDOUT_CASES<=len(rows), (
    f"Required TRAIN+DEV+HOLDOUT cases exceeds {len(rows)} dataset TRAIN cases")
train_rows=rows[:TRAIN_CASES]
dev_rows=rows[TRAIN_CASES:TRAIN_CASES+DEV_CASES]
holdout_rows=rows[TRAIN_CASES+DEV_CASES:TRAIN_CASES+DEV_CASES+HOLDOUT_CASES]
split_info=verify_case_partition(train_rows,dev_rows,holdout_rows)
probe_rows=dev_rows[:PROBE_CASES]
gate_rows=dev_rows[PROBE_CASES:]
verify_case_partition(train_rows,probe_rows,gate_rows)
assert not {str(x["id"]) for x in holdout_rows}&{
    str(x["id"]) for x in dev_rows}
train_items=make_items_from_rows(train_rows,SEED,True)
probe_items=make_items_from_rows(probe_rows,SEED+1,True)
gate_items=make_items_from_rows(gate_rows,SEED+2,True)
# IMPORTANT: holdout_items are NOT constructed yet. The final evaluation
# cell materializes these previously sealed rows after all decisions are made.
assert len(gate_items)>=DEV_GATE_DECISIONS
assert len(probe_items)>=PROBE_BATCHES*BATCH
print("Independent partitions:",split_info,
      "probe cases",len(probe_rows),"gate cases",len(gate_rows))
print("Optimizer/probe/gate decisions:",len(train_items),
      len(probe_items),len(gate_items))

def direct_batch(selected):
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k: v.to(device) for k, v in b.items()}

def batch(items, offset, n=BATCH):
    selected = [items[(offset + i) % len(items)] for i in range(n)]
    return direct_batch(selected)

# Only case-isolated TRAIN / PROBE / DEV-GATE inputs are available here.
# The independent HOLDOUT remains sealed until the final evaluation cell.
print("TRAIN / PROBE / DEV-GATE decisions:",
      len(train_items), len(probe_items), len(gate_items))
print("Official TEST untouched; independent HOLDOUT not materialized.")


README.md: 0.00B [00:00, ?B/s]

all/train-00000-of-00001.parquet:   0%|          | 0.00/599k [00:00<?, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/222k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1200 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/400 [00:00<?, ? examples/s]

Independent partitions: {'train_cases': 700, 'selection_cases': 220, 'holdout_cases': 200, 'all_partitions_disjoint': True} probe cases 80 gate cases 140
Optimizer/probe/gate decisions: 3500 400 700
TRAIN / PROBE / DEV-GATE decisions: 3500 400 700
Official TEST untouched; independent HOLDOUT not materialized.


In [3]:
#@title 3. Reuse V4/V5 substitutions or initialize all 28 from the original model
def unchanged_ffn_norm_hash(model):
    digest=hashlib.sha256()
    for i,layer in enumerate(model.encoder.layers):
        for name in ("mlp","mlp_norm","attn_norm"):
            for key,tensor in getattr(layer,name).state_dict().items():
                digest.update(f"{i}/{name}/{key}".encode())
                digest.update(tensor.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()

ORIGINAL_FFN_NORM_SHA=unchanged_ffn_norm_hash(student_eval)
def assert_frozen_originals():
    assert unchanged_ffn_norm_hash(student_eval)==ORIGINAL_FFN_NORM_SHA
    for layer in student_eval.encoder.layers:
        for name in ("mlp","mlp_norm","attn_norm"):
            assert all(not p.requires_grad for p in getattr(layer,name).parameters())
    assert all(not p.requires_grad for p in student_eval.head.parameters()) if student_eval.head is not None else True

# A loaded full V5 model can initialize V6 without its original per-layer ZIP.
layers_config=[]
source_mode=""
if INITIAL_HF_REPO.strip():
    print("Downloading trained all-28 V5 initialization:",INITIAL_HF_REPO)
    loaded=load_all28(INITIAL_HF_REPO.strip(),device="cpu")
    student_eval=loaded.model
    student_eval.eval().requires_grad_(False)
    student_eval.encoder.config.reference_compile=False
    assert unchanged_ffn_norm_hash(student_eval)==ORIGINAL_FFN_NORM_SHA
    layers_config=[{
        "layer":i,"window":a.window,"pattern":a.pattern,
        "anchors":a.anchors,"cenn_on":bool(a.use_cenn)}
        for i,layer in enumerate(student_eval.encoder.layers)
        for a in [layer.attn]]
    source_mode="v5_huggingface_full_model"
    del loaded
else:
    found={}
    candidate_dirs=[
        Path(V4_DIR),
        Path("/content"),
        Path("/content/Jev_CeNN_All28_HF_Export_V5/per_layer_checkpoints"),
    ]
    for i in range(28):
        filename=f"forced_best_attention_L{i:02d}.pt"
        for root in candidate_dirs:
            maybe=root/filename
            if maybe.is_file():
                found[i]=maybe
                break
    zip_path=Path(V4_ZIP_PATH)
    imported_dir=OUTPUT_DIR/"v4_imported"
    imported_dir.mkdir(exist_ok=True)
    if zip_path.is_file():
        with zipfile.ZipFile(zip_path) as archive:
            names={Path(x).name:x for x in archive.namelist()
                   if Path(x).name.startswith("forced_best_attention_L")
                   and Path(x).suffix==".pt"}
            for i in range(28):
                if i in found:continue
                filename=f"forced_best_attention_L{i:02d}.pt"
                if filename in names:
                    dest=imported_dir/filename
                    dest.write_bytes(archive.read(names[filename]))
                    found[i]=dest
    source_mode="v4_attention_checkpoints" if len(found)==28 else "partial_or_random_initialized"
    for i in range(28):
        original=student_eval.encoder.layers[i].attn
        if i in found:
            item=torch.load(found[i],map_location="cpu",weights_only=True)
            assert int(item["layer"])==i
            w,pattern,anchors=item["config"]
            module=WindowCeNNAttention(original,int(w),str(pattern),int(anchors))
            module.load_state_dict(item["state"],strict=True)
            original_enabled=bool(item["cenn_on"])
        else:
            # Fallback substitutes EVERY attention, no native layers retained.
            # Fidelity may be far below the prior V4 forced-model result.
            w,pattern,anchors=((385,"anchors",16)
                if i in GLOBAL_LAYERS else (97,"local",0))
            module=WindowCeNNAttention(original,w,pattern,anchors)
            original_enabled=False
        module.use_cenn=bool(ENABLE_CENN_ALL or original_enabled)
        module.eval().requires_grad_(False)
        student_eval.encoder.layers[i].attn=module
        layers_config.append({"layer":i,"window":int(w),
             "pattern":str(pattern),"anchors":int(anchors),
             "cenn_on":module.use_cenn})
if ENABLE_CENN_ALL:
    for layer in student_eval.encoder.layers:
        layer.attn.use_cenn=True
    for item in layers_config:item["cenn_on"]=True
assert len(layers_config)==28 and {x["layer"] for x in layers_config}==set(range(28))
assert all(isinstance(layer.attn,WindowCeNNAttention)
           for layer in student_eval.encoder.layers)
assert_frozen_originals()
print("Initialization:",source_mode,"layers",len(layers_config),
      "CeNN enabled",sum(x["cenn_on"] for x in layers_config))
print("FFN and norm SHA-256",ORIGINAL_FFN_NORM_SHA)


Initialization: partial_or_random_initialized layers 28 CeNN enabled 28
FFN and norm SHA-256 766166b2aa6376e9023fff6e2b63a2cb80618f2065597342f5cafe8be57670e6


In [4]:
#@title 4. Cache teacher logits ONCE, then release teacher GPU memory
teacher.to(device).eval().requires_grad_(False)
student_eval.to(device).eval()
# Use train cases only. No holdout examples are materialized before final run.
TRAIN_ITEMS=train_items[:min(TRAIN_DECISIONS_LIMIT,len(train_items))]
GATE_ITEMS=gate_items[:DEV_GATE_DECISIONS]
assert len(TRAIN_ITEMS)>=(BATCH*GRAD_ACCUM)
assert len(GATE_ITEMS)==DEV_GATE_DECISIONS

@torch.inference_mode()
def cache_teacher_prediction_records(items):
    records=[]
    for start in range(0,len(items),BATCH):
        source_items=items[start:start+BATCH]
        inp=direct_batch(source_items)
        with torch.autocast("cuda",dtype=amp_dtype,enabled=AMP):
            logits,_=teacher(**inp)
        records.append({
            "inputs":{k:v.detach().cpu().contiguous() for k,v in inp.items()},
            "teacher":logits.float().detach().cpu().contiguous(),
            "gold":torch.tensor([x["gold_index"] for x in source_items],
                                 dtype=torch.long)
        })
    return records

train_records=cache_teacher_prediction_records(TRAIN_ITEMS)
gate_records=cache_teacher_prediction_records(GATE_ITEMS)
print("Cached",len(train_records),"train batches and",
      len(gate_records),"independent DEV selection batches.")
teacher.cpu()
gc.collect()
torch.cuda.empty_cache()
assert_frozen_originals()

@torch.inference_mode()
def compare_with_teacher(model,records):
    model.eval()
    agreement=student_gold=teacher_gold=n=0
    js_total=0.0
    for rec in records:
        inp={k:v.to(device) for k,v in rec["inputs"].items()}
        valid=inp["marker_mask"].bool()
        with torch.autocast("cuda",dtype=amp_dtype,enabled=AMP):
            logits,_=model(**inp)
        t=rec["teacher"].to(device).masked_fill(~valid,-1e4)
        p=logits.float().masked_fill(~valid,-1e4)
        a=F.softmax(t,dim=-1)
        b=F.softmax(p,dim=-1)
        ap=a.argmax(-1);bp=b.argmax(-1)
        gold=rec["gold"].to(device)
        agreement+=int((ap==bp).sum())
        student_gold+=int((bp==gold).sum())
        teacher_gold+=int((ap==gold).sum())
        mid=0.5*(a+b)
        js_total+=float(0.5*(
            (a*(a.clamp_min(1e-9).log()-mid.clamp_min(1e-9).log())).sum(-1)
           +(b*(b.clamp_min(1e-9).log()-mid.clamp_min(1e-9).log())).sum(-1)
        ).sum())
        n+=len(ap)
    return {"decisions":n,"teacher_agreement":agreement/max(n,1),
       "student_gold_accuracy":student_gold/max(n,1),
       "teacher_gold_accuracy":teacher_gold/max(n,1),
       "mean_js":js_total/max(n,1)}

def cosine_lr(step,start,end,total=JOINT_STEPS):
    # At optimizer step 1, LR=start; at step JOINT_STEPS, LR=end.
    progress=(step-1)/max(1,total-1)
    if WARMUP_STEPS and step<=WARMUP_STEPS:
        # Optional warmup, strictly <=start. Exact 0.01 at step 1 if no warmup.
        return start*(step/max(1,WARMUP_STEPS))
    return end+0.5*(start-end)*(1+math.cos(math.pi*progress))


Cached 600 train batches and 120 independent DEV selection batches.


In [5]:
#@title 5. JOINT TRAINING all 28 attentions — CeNN LR 0.01 → 0.002
# FFNs/norms/decision heads remain bit-identical; no quality/acceptance gate.
cenn_params=[]
projection_params=[]
for i,layer in enumerate(student_eval.encoder.layers):
    a=layer.attn
    assert isinstance(a,WindowCeNNAttention) and a.use_cenn
    a.requires_grad_(False)
    # All 28 CeNN branches are optimized with requested LR.
    a.cenn.requires_grad_(True)
    a.cenn_gain.requires_grad_(True)
    cenn_params.extend(a.cenn.parameters())
    cenn_params.append(a.cenn_gain)
    # Extremely small LR for pretrained projections, optionally disabled.
    if TRAIN_PRETRAINED_QKV:
        a.Wqkv.requires_grad_(True)
        a.Wo.requires_grad_(True)
        projection_params+=list(a.Wqkv.parameters())+list(a.Wo.parameters())
student_eval.eval()
assert_frozen_originals()
assert len(cenn_params)>0
optimizer_groups=[{"params":cenn_params,"lr":LR_START,
                   "weight_decay":1e-5,"label":"cenn"}]
if projection_params:
    optimizer_groups.append({
        "params":projection_params,"lr":LR_START*PROJECTION_LR_SCALE,
        "weight_decay":1e-6,"label":"qkv"})
optimizer=torch.optim.AdamW(optimizer_groups,betas=(0.9,0.999),eps=1e-8)
scaler=torch.amp.GradScaler("cuda",enabled=use_scaler)
if ENABLE_GRADIENT_CHECKPOINTING:
    try:
        student_eval.encoder.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant":False})
        print("ModernBERT gradient checkpointing enabled")
    except Exception as exc:
        print("Gradient checkpointing unavailable; proceeding normally:",str(exc)[:140])
# Avoid warmup/profiling work with dropout. trainable parameters still receive grads.
BASELINE_DEV=compare_with_teacher(student_eval,gate_records)
print("Before V6 joint training:",BASELINE_DEV)
baseline_checkpoint=STAGE_DIR/"all28_initial_attention_trainable.pt"
def trainable_snapshot():
    return {key:value.detach().cpu().contiguous().clone()
            for key,value in student_eval.state_dict().items()
            if ".attn." in key and any(
                 segment in key for segment in
                 (".cenn.",".cenn_gain",".Wqkv.",".Wo."))}
def save_trainable(path,step):
    torch.save({"step":step,"state_dict":trainable_snapshot(),
                "lr_cenn":cosine_lr(max(1,step),LR_START,LR_END)},
               path)
save_trainable(baseline_checkpoint,0)

history=[]
order=list(range(len(train_records)))
rng=random.Random(SEED+61)
record_index=0
start_wall=time.perf_counter()
for step in range(1,JOINT_STEPS+1):
    # HF GradientCheckpointingLayer checkpoints only in encoder TRAIN mode.
    # Keep Dropout modules in eval mode for deterministic teacher KD.
    if ENABLE_GRADIENT_CHECKPOINTING:
        student_eval.encoder.train()
        for submodule in student_eval.encoder.modules():
            if isinstance(submodule,torch.nn.Dropout):
                submodule.eval()
    lr=cosine_lr(step,LR_START,LR_END)
    optimizer.param_groups[0]["lr"]=lr
    if projection_params:
        optimizer.param_groups[1]["lr"]=lr*PROJECTION_LR_SCALE
    optimizer.zero_grad(set_to_none=True)
    sums={"loss":0.0,"kl":0.0,"ce":0.0}
    for acc in range(GRAD_ACCUM):
        if record_index==0:rng.shuffle(order)
        rec=train_records[order[record_index]]
        record_index=(record_index+1)%len(order)
        inp={key:value.to(device,non_blocking=True)
             for key,value in rec["inputs"].items()}
        gold=rec["gold"].to(device)
        ref_logits=rec["teacher"].to(device)
        with torch.autocast("cuda",dtype=amp_dtype,enabled=AMP):
            student_logits,_=student_eval(**inp)
            mask=inp["marker_mask"].bool()
            t=ref_logits.float().masked_fill(~mask,-1e4)
            s=student_logits.float().masked_fill(~mask,-1e4)
            temp=TEACHER_TEMPERATURE
            kl=F.kl_div(
                F.log_softmax(s/temp,dim=-1),
                F.softmax(t/temp,dim=-1),
                reduction="batchmean")*(temp**2)
            ce=F.cross_entropy(s,gold)
            loss=KL_WEIGHT*kl+GOLD_WEIGHT*ce
            loss_scaled=loss/GRAD_ACCUM
        require_finite(loss,"joint substitution KD")
        scaler.scale(loss_scaled).backward()
        sums["loss"]+=float(loss.detach())/GRAD_ACCUM
        sums["kl"]+=float(kl.detach())/GRAD_ACCUM
        sums["ce"]+=float(ce.detach())/GRAD_ACCUM
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(cenn_params+projection_params,
                                   max_norm=MAX_GRAD_NORM)
    scaler.step(optimizer)
    scaler.update()
    if step==1 or step%EVAL_EVERY==0 or step==JOINT_STEPS:
        dev=compare_with_teacher(student_eval,gate_records)
        record={"step":step,"cenn_lr":lr,
          "projection_lr":lr*PROJECTION_LR_SCALE if projection_params else 0,
          "train":sums,"dev":dev}
        history.append(record)
        (OUTPUT_DIR/"training_progress.json").write_text(
            json.dumps({"initial":BASELINE_DEV,"history":history},indent=2))
        print(f"step {step}/{JOINT_STEPS} LR={lr:.6f} "
              f"KD={sums['kl']:.5f} CE={sums['ce']:.5f} "
              f"DEV teacher={dev['teacher_agreement']:.4f} "
              f"gold={dev['student_gold_accuracy']:.4f}")
    if step%SAVE_EVERY==0 or step==JOINT_STEPS:
        save_trainable(STAGE_DIR/f"all28_attention_step_{step:04d}.pt",step)
assert abs(cosine_lr(1,LR_START,LR_END)-LR_START)<1e-10
assert abs(cosine_lr(JOINT_STEPS,LR_START,LR_END)-LR_END)<1e-10
student_eval.eval()
if ENABLE_GRADIENT_CHECKPOINTING:
    try:student_eval.encoder.gradient_checkpointing_disable()
    except Exception:pass
FINAL_DEV=compare_with_teacher(student_eval,gate_records)
assert_frozen_originals()
print("Finished all-28 joint substitution training:",FINAL_DEV)
print("Training seconds",round(time.perf_counter()-start_wall,1))
del optimizer
gc.collect();torch.cuda.empty_cache()


ModernBERT gradient checkpointing enabled
Before V6 joint training: {'decisions': 240, 'teacher_agreement': 0.9416666666666667, 'student_gold_accuracy': 0.8208333333333333, 'teacher_gold_accuracy': 0.8416666666666667, 'mean_js': 0.002208278397468651}
step 1/300 LR=0.010000 KD=0.00599 CE=0.53372 DEV teacher=0.9417 gold=0.8208
step 50/300 LR=0.009481 KD=0.17076 CE=0.57694 DEV teacher=0.9083 gold=0.7958
step 100/300 LR=0.008024 KD=0.01403 CE=0.69318 DEV teacher=0.9042 gold=0.7750
step 150/300 LR=0.006021 KD=0.00901 CE=0.57951 DEV teacher=0.9333 gold=0.8042
step 200/300 LR=0.004012 KD=0.00755 CE=0.68301 DEV teacher=0.9167 gold=0.8125
step 250/300 LR=0.002539 KD=0.02239 CE=0.52724 DEV teacher=0.9083 gold=0.7875
step 300/300 LR=0.002000 KD=0.02929 CE=0.57810 DEV teacher=0.9333 gold=0.8125
Finished all-28 joint substitution training: {'decisions': 240, 'teacher_agreement': 0.9333333333333333, 'student_gold_accuracy': 0.8125, 'teacher_gold_accuracy': 0.8416666666666667, 'mean_js': 0.0030405468

In [6]:
#@title 6. ONE-TIME holdout audit + paired full-model latency (no selection gate)
# Do not use the holdout to choose training steps or learning rates.
teacher.to(device).eval()
HOLDOUT_ITEMS=make_items_from_rows(holdout_rows,SEED+999,False)[:HOLDOUT_DECISIONS]
assert len(HOLDOUT_ITEMS)==HOLDOUT_DECISIONS
holdout_records=cache_teacher_prediction_records(HOLDOUT_ITEMS)
teacher.cpu();torch.cuda.empty_cache()
FINAL_HOLDOUT=compare_with_teacher(student_eval,holdout_records)
assert_frozen_originals()
print("Final V6 untouched holdout:",FINAL_HOLDOUT)
@torch.inference_mode()
def latency(model,inp,repeats=BENCH_REPEATS):
    model.eval()
    for _ in range(4):model(**inp)
    torch.cuda.synchronize()
    measurements=[]
    for _ in range(repeats):
        a=torch.cuda.Event(enable_timing=True)
        b=torch.cuda.Event(enable_timing=True)
        a.record()
        model(**inp)
        b.record();b.synchronize()
        measurements.append(float(a.elapsed_time(b)))
    return {"median_ms":float(np.median(measurements)),
        "p95_ms":float(np.percentile(measurements,95)),
        "repeats":repeats}
teacher.to(device).eval()
if ENABLE_GRADIENT_CHECKPOINTING:
    try:student_eval.encoder.gradient_checkpointing_disable()
    except Exception:pass
SHORT=[e for e in gate_items+probe_items if len(e["ids"])<=160]
LONG=[e for e in gate_items+probe_items if 384<=len(e["ids"])<=MAX_LEN]
timings=[]
for label,pool in (("short",SHORT),("long",LONG)):
    for n in (1,2):
        if len(pool)<n:
            timings.append({"type":label,"batch":n,"status":"missing_real_samples"})
            continue
        inp=direct_batch(pool[:n])
        a=latency(teacher,inp)
        b=latency(student_eval,inp)
        timings.append({"type":label,"batch":n,
          "lengths":[len(x["ids"]) for x in pool[:n]],
          "teacher":a,"substitute":b,
          "speedup":a["median_ms"]/max(1e-9,b["median_ms"]),
          "status":"measured"})
teacher.cpu();torch.cuda.empty_cache()
print("Paired end-to-end timings:",json.dumps(timings,indent=2))


Final V6 untouched holdout: {'decisions': 500, 'teacher_agreement': 0.93, 'student_gold_accuracy': 0.858, 'teacher_gold_accuracy': 0.864, 'mean_js': 0.005485029115166981}
Paired end-to-end timings: [
  {
    "type": "short",
    "batch": 1,
    "lengths": [
      136
    ],
    "teacher": {
      "median_ms": 56.04185485839844,
      "p95_ms": 56.8555305480957,
      "repeats": 15
    },
    "substitute": {
      "median_ms": 74.24607849121094,
      "p95_ms": 77.97055358886719,
      "repeats": 15
    },
    "speedup": 0.7548123213676872,
    "status": "measured"
  },
  {
    "type": "short",
    "batch": 2,
    "lengths": [
      136,
      147
    ],
    "teacher": {
      "median_ms": 104.14694213867188,
      "p95_ms": 105.46630401611328,
      "repeats": 15
    },
    "substitute": {
      "median_ms": 138.8797149658203,
      "p95_ms": 140.4096923828125,
      "repeats": 15
    },
    "speedup": 0.7499075164742632,
    "status": "measured"
  },
  {
    "type": "long",
    "batch

In [7]:
#@title 7. Export NEW complete V6 model, exact FFNs, tokenizer, manifest
# Torch serialization uses one standard .safetensors file for all learned
# projections, CeNN parameters, original pretrained FFNs and decision head.
student_eval.cpu().eval().requires_grad_(False)
assert_frozen_originals()
model_path=EXPORT_DIR/"model.safetensors"
save_model(student_eval,str(model_path))
tokenizer.save_pretrained(str(EXPORT_DIR/"tokenizer"))
(EXPORT_DIR/"encoder").mkdir(exist_ok=True)
shutil.copy2(source_dir/"encoder"/"config.json",
             EXPORT_DIR/"encoder"/"config.json")
shutil.copy2(source_dir/"rl_agent_config.json",
             EXPORT_DIR/"rl_agent_config.json")
shutil.copy2(Path(model_code.__file__),EXPORT_DIR/"all28_hf.py")
manifest={
 "architecture":"jev_cenn_all28_joint_substitution_v6",
 "source_model":SOURCE_MODEL,
 "initialized_from":INITIAL_HF_REPO if INITIAL_HF_REPO.strip() else source_mode,
 "attention_layers":layers_config,
 "original_encoder_layers":28,
 "converted_attention_layers":list(range(28)),
 "original_ffns_preserved":True,
 "ffn_norm_sha256":ORIGINAL_FFN_NORM_SHA,
 "training_lr_start":LR_START,
 "training_lr_end":LR_END,
 "training_steps":JOINT_STEPS,
 "projection_lr_scale":PROJECTION_LR_SCALE,
 "gold_weight":GOLD_WEIGHT,
 "teacher_kl_weight":KL_WEIGHT,
 "joint_training":True,
 "quality_gate_required":False,
 "trained_utc":datetime.now(timezone.utc).isoformat(),
 "baseline_dev":BASELINE_DEV,
 "final_dev":FINAL_DEV,
 "final_holdout":FINAL_HOLDOUT,
 "speed":timings,
 "parameter_count":sum(p.numel() for p in student_eval.parameters())
}
(EXPORT_DIR/"all28_manifest.json").write_text(
  json.dumps(manifest,indent=2,allow_nan=False),encoding="utf-8")
readme=f"""---
library_name: pytorch
tags:
- cenn
- modernbert
- all-28-attention
- decision-model
- knowledge-distillation
---

# Jev-CeNN All-28 Substitution V6 (LR 0.01 → 0.002)

Full 28-layer encoder with all 28 pretrained attention blocks substituted,
original FFNs and LayerNorms preserved, and a frozen task head. Trained jointly
with KL teacher distillation and gold-label CE on the original typed-decisions task.

- CeNN LR: **{LR_START} → {LR_END}** via cosine schedule over {JOINT_STEPS} steps
- QKV/output projections: **{LR_START*PROJECTION_LR_SCALE:.8f} → {LR_END*PROJECTION_LR_SCALE:.8f}**
- DEV teacher agreement: **{FINAL_DEV['teacher_agreement']:.4f}**
- Holdout teacher agreement: **{FINAL_HOLDOUT['teacher_agreement']:.4f}**
- Holdout gold accuracy: **{FINAL_HOLDOUT['student_gold_accuracy']:.4f}**
- All original FFNs/norms unchanged SHA-256: `{ORIGINAL_FFN_NORM_SHA}`
- No fidelity acceptance gate. Speedup is not guaranteed.

## Install and load
```bash
pip install 'transformers==4.57.6' safetensors huggingface_hub
pip install 'git+https://github.com/vtavakkoli/TinyCeNN-LM.git@eba220d8131dbdf06f748545a21831327e5ad37d'
```

```python
from tinycenn_lm.all28_hf import load_all28
runtime=load_all28('REPLACE_WITH_YOUR_HF_ID',device='cuda')
result=runtime.decide(
    {{'road_direction':'left','speed':'7.0'}},
    ['left','straight','right'])
print(result)
```

The model is a custom `StandaloneDecisionModel`, not an autoregressive
chat model. Hugging Face `AutoModel.from_pretrained` is not supported:
use `all28_hf.py` or the installed `tinycenn_lm.all28_hf` loader.

Source checkpoint: {SOURCE_MODEL}. Please verify redistribution licensing
before changing this repository from private to public.
"""
(EXPORT_DIR/"README.md").write_text(readme,encoding="utf-8")
(EXPORT_DIR/"requirements.txt").write_text(
    "transformers==4.57.6\nsafetensors>=0.4\nhuggingface_hub>=0.25,<1\n"
    "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@eba220d8131dbdf06f748545a21831327e5ad37d\n"
)
(OUTPUT_DIR/"final_report.json").write_text(
    json.dumps({"manifest":manifest,"history":history},indent=2,allow_nan=False))
print("New FULL model exported:",model_path,round(model_path.stat().st_size/2**20,1),"MiB")


New FULL model exported: /content/Jev_CeNN_All28_Substitution_V6/huggingface_model/model.safetensors 2058.6 MiB


In [8]:
#@title 8. Verify full-model local round-trip before Hugging Face publication
runtime=build_all28_from_dir(EXPORT_DIR,device="cpu")
loaded=runtime.model
assert len(loaded.encoder.layers)==28
assert all(getattr(a.attn,"architecture",None)==WindowCeNNAttention.architecture
           for a in loaded.encoder.layers)
assert unchanged_ffn_norm_hash(loaded)==ORIGINAL_FFN_NORM_SHA
sample=tokenizer("The agent must decide whether to turn left at this intersection.",
                 return_tensors="pt",max_length=64,truncation=True)
p=sample["input_ids"].shape[1]
assert p>=2
smoke={
 "input_ids":sample["input_ids"],
 "attention_mask":sample["attention_mask"],
 "marker_pos":torch.tensor([[0,1]],dtype=torch.long),
 "marker_mask":torch.ones((1,2),dtype=torch.bool),
 "qtype":torch.tensor([0],dtype=torch.long)
}
with torch.inference_mode():
    a,_=student_eval(**smoke)
    b,_=loaded(**smoke)
    error=float((a.float()-b.float()).abs().max())
assert error<1e-4,("Standalone roundtrip mismatch",error)
print("FULL HF export reload parity PASSED; max logit difference",error)
print("All 28 ATTENTION modules present, original FFNs preserved.")
del runtime,loaded
gc.collect()


FULL HF export reload parity PASSED; max logit difference 0.0
All 28 ATTENTION modules present, original FFNs preserved.


0

In [9]:
#@title 9. Publish NEW full model to Hugging Face (private by default)
if UPLOAD_TO_HF:
    token=get_token()
    if not token:
        login()  # secure interactive Hugging Face authorization
        token=get_token()
    if not token:raise RuntimeError("Hugging Face write token required")
    api=HfApi(token=token)
    account=api.whoami(token=token)
    namespace=account["name"]
    repo_id=f"{namespace}/{HF_REPO_NAME}"
    card=EXPORT_DIR/"README.md"
    card.write_text(card.read_text(encoding="utf-8").replace(
        "REPLACE_WITH_YOUR_HF_ID",repo_id),encoding="utf-8")
    api.create_repo(repo_id=repo_id,repo_type="model",private=HF_PRIVATE,
                    exist_ok=True)
    api.upload_folder(repo_id=repo_id,repo_type="model",
                      folder_path=str(EXPORT_DIR),
                      commit_message=f"Full all-28 substitution model, cosine CeNN LR {LR_START} to {LR_END}")
    files=set(api.list_repo_files(repo_id=repo_id,repo_type="model"))
    expected={"model.safetensors","all28_hf.py","all28_manifest.json",
              "encoder/config.json","rl_agent_config.json","README.md"}
    assert expected<=files,sorted(expected-files)
    print("New model published:",f"https://huggingface.co/{repo_id}")
else:
    print("HF upload disabled. Complete model in:",EXPORT_DIR)


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...e_model/model.safetensors:   2%|1         | 40.0MB / 2.16GB            

New model published: https://huggingface.co/vtava/jev-cenn-all28-substitution-lr001-lr002-v6
